In [ ]:
import numpy as np

path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/checkpoints/class_1/ksvd_checkpoint.npz"

data = np.load(path)

print(data.files)  # list stored arrays

for key in data.files:
    print(key, data[key].shape)

In [ ]:
print(data["completed_iter"])
print(data["errors_"])

In [ ]:
import numpy as np
import pickle

# Path to trained dictionary
dict_path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/models/unified_frozen.pkl"

# Load dictionary model
with open(dict_path, "rb") as f:
    model_data = pickle.load(f)

# Extract dictionary matrix
D = model_data["model"].D_

print("Dictionary loaded successfully")
print("Dictionary shape:", D.shape)
print("Dictionary dtype:", D.dtype)

# Load sparse codes
path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/train_sparse_codes.npz"

data = np.load(path)

Gamma = data["Gamma"]
labels = data["labels"]

print("Gamma shape:", Gamma.shape)
print("Labels shape:", labels.shape)

classes = np.unique(labels)

# Atom Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

D = pickle.load(
    open(
        "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/models/unified_frozen.pkl",
        "rb"
    ))["model"].D_

PATCH_SIZE = 12
MIDDLE = PATCH_SIZE // 2

rng = np.random.default_rng(42)

GRID_SIZE = 40
N_ATOMS = GRID_SIZE * GRID_SIZE  # 900

normal_atoms = rng.choice(
    np.arange(0, 6812),
    size=N_ATOMS,
    replace=False
)

ggo_atoms = rng.choice(
    np.arange(6912, 10368),
    size=N_ATOMS,
    replace=False
)

nodule_atoms = rng.choice(
    np.arange(10368, 12096),
    size=N_ATOMS,
    replace=False
)

def plot_grid(fig, left, right, atom_ids, title):

    gs = fig.add_gridspec(
        GRID_SIZE,
        GRID_SIZE,
        left=left,
        right=right,
        top=0.88,
        bottom=0.02,
        wspace=0,
        hspace=0
    )

    axes = []

    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            ax = fig.add_subplot(gs[i, j])
            ax.axis("off")
            axes.append(ax)

    fig.text(
        (left + right) / 2,
        0.93,
        title,
        ha="center",
        fontsize=16
    )

    for ax, atom_id in zip(axes, atom_ids):
        atom = D[:, atom_id].reshape(
            PATCH_SIZE,
            PATCH_SIZE,
            PATCH_SIZE
        )

        ax.imshow(atom[MIDDLE], cmap="gray")
        ax.axis("off")


fig = plt.figure(figsize=(20, 7))


# Normal atoms
plot_grid(
    fig,
    0.01,
    0.33,
    normal_atoms,
    "Normal Atoms"
)


# GGO atoms
plot_grid(
    fig,
    0.34,
    0.66,
    ggo_atoms,
    "GGO Atoms"
)


# Nodule atoms
plot_grid(
    fig,
    0.67,
    0.99,
    nodule_atoms,
    "Nodule Atoms"
)


plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Dictionary shape:", D.shape)

reconstructed_patches = {}

# Create one figure for all classes
fig, axes = plt.subplots(
    1,
    len(classes),
    figsize=(15,5)
)

for i, cls in enumerate(classes):

    Gamma_cls = Gamma[:, labels == cls]

    # Mean absolute coefficient values
    mean_abs_coefficients = np.mean(
        np.abs(Gamma_cls),
        axis=1
    )

    # Top 10 coefficient indices
    top10_idx = np.argsort(
        mean_abs_coefficients
    )[::-1][:10]


    # Average signed coefficient values
    top10_values = np.mean(
        Gamma_cls[top10_idx, :],
        axis=1
    )


    print("\n==============================")
    print(f"Class {cls}")
    print("Top coefficients")

    for idx, value in zip(top10_idx, top10_values):
        print(
            f"Index {idx:5d} | Mean coefficient {value:.6f}"
        )


    # Reconstruction
    D_selected = D[:, top10_idx]

    reconstructed = D_selected @ top10_values

    reconstructed = reconstructed.reshape(12,12,12)

    reconstructed_patches[cls] = reconstructed


    print("Reconstructed patch shape:", reconstructed.shape)


    # Middle slice
    mid = reconstructed.shape[2] // 2

    axes[i].imshow(
        reconstructed[:, :, mid],
        cmap="gray"
    )

    axes[i].set_title(
        f"Class {cls}"
    )

    axes[i].axis("off")


plt.suptitle(
    "Class-wise Reconstruction from Top-10 Sparse Coefficients",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_3d_patch_subplot(ax, volume, title, threshold=None):

    if threshold is None:
        threshold = np.percentile(
            np.abs(volume),
            70
        )

    mask = np.abs(volume) > threshold

    x, y, z = np.where(mask)
    values = volume[mask]

    ax.scatter(
        x,
        y,
        z,
        s=60,
        c=values,
        cmap="gray"
    )

    ax.set_title(title)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")


# Create one figure
fig = plt.figure(figsize=(18,6))

for i, cls in enumerate([0, 1, 2]):

    patch = reconstructed_patches[cls]

    ax = fig.add_subplot(
        1,
        3,
        i+1,
        projection="3d"
    )

    plot_3d_patch_subplot(
        ax,
        patch,
        title=f"Class {cls}"
    )


plt.suptitle(
    "3D Reconstruction from Top-10 Sparse Coefficients",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

with open(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/models/unified_frozen.pkl",
    "rb"
) as f:
    model_data = pickle.load(f)

ksvd = model_data["model"]
D = ksvd.D_

sparse_codes = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/train_sparse_codes_frozen.npz",
    allow_pickle=True
)
print(sparse_codes.files)

patch_data = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_2d.npz",
    allow_pickle=True
)

X = patch_data["X"]
scan_ids = patch_data["scan_ids"]
coords = patch_data["coords"]
H = patch_data["H"]

target_scan = "train_11150_b_2"

Gamma = sparse_codes["Gamma"]
labels = sparse_codes["labels"]

# Find the patches that actually belong to the target scan
indices = np.where(scan_ids == target_scan)[0]

print("Indices:")
print(indices)

print("\nCoordinates:")
print(coords[indices])

# Reconstruct exactly those patches, using the same indices
gamma = Gamma[:, indices]
reconstructed = D @ gamma

print(reconstructed.shape)

patches = reconstructed.T.reshape(-1, 12, 12, 12)
print(patches.shape)

min_coord = coords[indices].min(axis=0)
max_coord = coords[indices].max(axis=0)

shape = tuple(max_coord - min_coord + 12)

volume = np.zeros(shape)
weight = np.zeros(shape)

for patch, coord in zip(patches, coords[indices]):
    x, y, z = coord - min_coord
    volume[x:x+12, y:y+12, z:z+12] += patch
    weight[x:x+12, y:y+12, z:z+12] += 1

volume /= np.maximum(weight, 1)

# Visualize the reconstructed volume
cx, cy, cz = np.array(volume.shape) // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(volume[:, :, cz], cmap="gray")
ax[0].set_title("Axial")

ax[1].imshow(volume[:, cy, :], cmap="gray")
ax[1].set_title("Coronal")

ax[2].imshow(volume[cx, :, :], cmap="gray")
ax[2].set_title("Sagittal")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

with open(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/models/unified_frozen.pkl",
    "rb"
) as f:
    model = pickle.load(f)

ksvd = model["model"]
D = ksvd.D_


# Take top 4 atoms from previous calculation
top_4_atoms = top_10_atoms[:10]

print("Top 4 atoms:", top_4_atoms)


# Aggregate coefficient for each selected atom
# (sum of signed coefficients across all patches)
atom_coefficients = np.sum(
    Gamma_patches[top_4_atoms, :],
    axis=1
)

print("\nAtom coefficients:")
for atom, coef in zip(top_4_atoms, atom_coefficients):
    print(f"Atom {atom}: {coef:.6f}")


# Reconstruct contribution of each atom
# D[:, atom] * coefficient
atom_contributions = []

for atom, coef in zip(top_4_atoms, atom_coefficients):

    patch = D[:, atom] * coef

    atom_contributions.append(
        patch
    )


# Superimpose the 4 atom contributions
combined_patch = np.sum(
    atom_contributions,
    axis=0
)


print("\nCombined patch shape:", combined_patch.shape)


# Convert to 3D
combined_patch_3d = combined_patch.reshape(
    16,16,16
)


# Visualize
mid = 8

fig, axes = plt.subplots(
    1,3,
    figsize=(12,4)
)

axes[0].imshow(
    combined_patch_3d[:,:,mid],
    cmap="gray"
)
axes[0].set_title("Axial")

axes[1].imshow(
    combined_patch_3d[:,mid,:],
    cmap="gray"
)
axes[1].set_title("Coronal")

axes[2].imshow(
    combined_patch_3d[mid,:,:],
    cmap="gray"
)
axes[2].set_title("Sagittal")


for ax in axes:
    ax.axis("off")

plt.suptitle(
    "Reconstruction from Top 4 Dictionary Atoms"
)

plt.show()

# Reconstruction

In [ ]:
ksvd = model["model"]
D = ksvd.D_

idx = 97991
gamma = Gamma[:, idx]

reconstructed_patch = D @ gamma

print("Dictionary shape:", D.shape)
print("Reconstructed patch shape:", reconstructed_patch.shape)

patch_3d = reconstructed_patch.reshape(16, 16, 16)

import matplotlib.pyplot as plt

# Choose center slices
mid = 8

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Axial view
axes[0].imshow(
    patch_3d[:, :, mid],
    cmap="gray"
)
axes[0].set_title("Axial")

# Coronal view
axes[1].imshow(
    patch_3d[:, mid, :],
    cmap="gray"
)
axes[1].set_title("Coronal")

# Sagittal view
axes[2].imshow(
    patch_3d[mid, :, :],
    cmap="gray"
)
axes[2].set_title("Sagittal")

for ax in axes:
    ax.axis("off")

plt.suptitle("Reconstructed Patch from LC-KSVD Sparse Code")
plt.show()

In [ ]:
# Assumed sparse code indices
gamma_indices = np.arange(11751, 11762)

# Extract the 12 sparse codes
gamma = Gamma[:, gamma_indices]
ksvd = model["model"]
D = ksvd.D_
# Reconstruct all patches
reconstructed = D @ gamma

print(reconstructed.shape)

patches = reconstructed.T.reshape(-1, 12, 12, 12)

print(patches.shape)

coords12 = np.array([
    [61, 158, 116],
    [61, 158, 120],
    [65, 154, 116],
    [65, 154, 120],
    [65, 158, 112],
    [65, 158, 116],
    [65, 158, 120],
    [65, 162, 116],
    [65, 162, 120],
    [69, 158, 116],
    [69, 158, 120],
    [69, 162, 116]
])

min_coord = coords12.min(axis=0)
max_coord = coords12.max(axis=0)

shape = tuple(max_coord - min_coord + 16)

volume = np.zeros(shape)
weight = np.zeros(shape)

for patch, coord in zip(patches, coords12):

    x, y, z = coord - min_coord

    volume[x:x+12, y:y+12, z:z+12] += patch
    weight[x:x+12, y:y+12, z:z+12] += 1

volume /= np.maximum(weight, 1)

# Visualize the reconstructed volume

import matplotlib.pyplot as plt

cx, cy, cz = np.array(volume.shape) // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(volume[:, :, cz], cmap="gray")
ax[0].set_title("Axial")

ax[1].imshow(volume[:, cy, :], cmap="gray")
ax[1].set_title("Coronal")

ax[2].imshow(volume[cx, :, :], cmap="gray")
ax[2].set_title("Sagittal")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()